In [ ]:
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from snail.damage_library import get_metadata, load_curve
from snail.damages import calculate_direct_damages
from snail.overlay import overlay_rasters

# Support running Jupyter from either the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
road_path = repo_root / "data/lca/asset/Roads_rmms_2018.gpkg"
flood_dir = repo_root / "data/lca/hazard/flood"
measurement_crs = "EPSG:32620"

# Unlabelled present-day rasters are GWL1.3; named GWLs come from filenames.
filename_pattern = re.compile(
    r"^LC_\d{4}_FLRF_U_(?:GWL(?P<gwl>\d+(?:[p.]\d+)?)_)?"
    r"RP(?P<return_period>\d+)_RD_30m_4326\.tif$"
)
raster_records = []
for raster_path in sorted(flood_dir.glob("*.tif")):
    match = filename_pattern.fullmatch(raster_path.name)
    if match is None:
        continue
    gwl = float((match.group("gwl") or "1.3").replace("p", "."))
    raster_records.append(
        {
            "scenario": f"GWL{gwl:g}",
            "gwl": gwl,
            "return_period": int(match.group("return_period")),
            "path": raster_path,
        }
    )
if not raster_records:
    raise ValueError(f"No matching flood rasters found in {flood_dir}")
raster_manifest = pd.DataFrame(raster_records).sort_values(["gwl", "return_period"])
if raster_manifest.duplicated(["scenario", "return_period"]).any():
    raise ValueError("Found more than one raster for a scenario/return-period pair")
flood_return_periods = tuple(sorted(raster_manifest["return_period"].unique()))
scenario_levels = raster_manifest[["scenario", "gwl"]].drop_duplicates()
for scenario, records in raster_manifest.groupby("scenario", sort=False):
    if set(records["return_period"]) != set(flood_return_periods):
        raise ValueError(f"Incomplete return-period coverage for {scenario}")
print(f"Found {len(raster_manifest)} rasters for {len(scenario_levels)} GWL scenarios.")
raster_manifest.groupby("scenario", sort=False).agg(
    gwl=("gwl", "first"), return_periods=("return_period", list)
)

In [ ]:
# Load every road feature, including roads with no modelled inundation.
roads = gpd.read_file(road_path, layer="Roads_RMMS_2018")
if not roads["ogc_fid"].is_unique:
    raise ValueError("Expected a unique ogc_fid for every road asset")
road_length_km = roads.to_crs(measurement_crs).length.sum() / 1_000
curve_metadata = {curve_id: get_metadata(curve_id) for curve_id in ("F7.2", "F7.3")}
curves = {curve_id: load_curve(curve_id) for curve_id in curve_metadata}

print(f"Road assets: {len(roads):,}; total length: {road_length_km:,.1f} km")
for curve_id, metadata in curve_metadata.items():
    print(
        f"{curve_id}: {metadata.hazard_name} hazard, "
        f"{metadata.exposed_element}, intensity unit {metadata.intensity_unit}"
    )
roads.head()

In [ ]:
scenario_frames = []
coverage_rows = []
intensity_columns = {f"flood_rp{rp}": f"flood_rp{rp}" for rp in flood_return_periods}

# Split separately by scenario: the present-day and future raster transforms
# differ slightly, so overlaying all 30 at once would create unnecessary slivers.
for scenario, records in raster_manifest.groupby("scenario", sort=False):
    scenario_rasters = pd.DataFrame(
        {
            "path": records["path"].tolist(),
            "key": [f"flood_rp{rp}" for rp in records["return_period"]],
        }
    )
    scenario_splits = overlay_rasters(roads[["ogc_fid", "geometry"]], scenario_rasters)
    scenario_splits["length_km"] = (
        scenario_splits.geometry.to_crs(measurement_crs).length / 1_000
    )
    scenario_splits["scenario"] = scenario
    assessed_length_km = scenario_splits["length_km"].sum()
    if scenario_splits["ogc_fid"].nunique() != len(roads):
        raise ValueError(f"Some road assets were lost during splitting for {scenario}")
    if not np.isclose(assessed_length_km, road_length_km, rtol=1e-5):
        raise ValueError(
            f"Road length was not conserved during splitting for {scenario}"
        )

    for record in records.itertuples():
        case = f"flood_rp{record.return_period}"
        with rasterio.open(record.path) as raster:
            nodata = raster.nodata
        depth = scenario_splits[case]
        missing = ~np.isfinite(depth)
        if nodata is not None:
            missing |= depth == nodata
        if ((depth < 0) & ~missing).any():
            raise ValueError(f"Unexpected negative flood depth in {record.path.name}")
        scenario_splits[case] = depth.where(~missing, 0.0)
        flooded = scenario_splits[case] > 0
        coverage_rows.append(
            {
                "scenario": scenario,
                "return_period": record.return_period,
                "n_assets": scenario_splits["ogc_fid"].nunique(),
                "assessed_length_km": assessed_length_km,
                "flooded_length_km": scenario_splits.loc[flooded, "length_km"].sum(),
            }
        )
    scenario_frames.append(
        pd.DataFrame(
            scenario_splits[["ogc_fid", "split", "scenario", "length_km", *intensity_columns]]
        )
    )
damage_splits = pd.concat(scenario_frames, ignore_index=True)
coverage = pd.DataFrame(coverage_rows)
coverage.round(2)

In [ ]:
result = calculate_direct_damages(
    damage_splits,
    asset_id="ogc_fid",
    scenario="scenario",
    quantity="length_km",
    intensity_columns=intensity_columns,
    return_periods={f"flood_rp{rp}": float(rp) for rp in flood_return_periods},
    curves=curves,
    curve_weights={"F7.2": 0.3, "F7.3": 0.7},
    cost_mean=300_000,
    cost_systemic_cv=0.25,
    hazard_cv={case: 0.1 for case in intensity_columns},
    eta_sd=0.1,
    n_samples=1_000,
    quantiles=None,
    ead_quantiles=(0.05, 0.25, 0.5, 0.75, 0.95),
    seed=42,
    aggregate_assets=True,
)

total_ead = (
    result.total_ead_quantiles.merge(
        scenario_levels, on="scenario", validate="one_to_one"
    )
    .sort_values("gwl")
    .reset_index(drop=True)
)
if not (total_ead["n_assets"] == len(roads)).all():
    raise ValueError("The total EAD results do not include every road asset")
total_ead[
    ["scenario", "gwl", "n_assets", "n_splits", "q_05", "q_25", "q_50", "q_75", "q_95"]
]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = total_ead["gwl"].to_numpy()
ax.fill_between(
    x,
    total_ead["q_05"].to_numpy(),
    total_ead["q_95"].to_numpy(),
    color="tab:blue",
    alpha=0.15,
    label="5th–95th percentiles",
)
ax.fill_between(
    x,
    total_ead["q_25"].to_numpy(),
    total_ead["q_75"].to_numpy(),
    color="tab:blue",
    alpha=0.3,
    label="25th–75th percentiles",
)
ax.plot(x, total_ead["q_50"].to_numpy(), marker="o", color="tab:blue", label="Median")
ax.set_xticks(x, total_ead["scenario"])
ax.set(
    xlabel="Global warming level (°C)",
    ylabel="Total expected annual damage (currency units/year)",
    title=f"Whole road network EAD across warming levels ({len(roads):,} assets)",
    ylim=(0, None),
)
ax.ticklabel_format(axis="y", style="plain", useOffset=False)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()